# Week 4 Solution — Python → C++ Code Generator

Use frontier AI models to convert Python code into high-performance C++ code, then compile and execute it locally.

**Requirements covered:**
- Python → C++ code generation via multiple frontier models
- Local compilation & execution with subprocess
- Performance benchmarking and speedup comparison
- Robust error handling with graceful fallbacks
- Interactive Gradio UI for live code conversion

## 1. Setup

In [11]:
import os
import io
import sys
import time
import subprocess
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr
from IPython.display import Markdown, display

load_dotenv(override=True)

openai_api_key     = os.getenv('OPENAI_API_KEY')
anthropic_api_key  = os.getenv('ANTHROPIC_API_KEY')
google_api_key     = os.getenv('GOOGLE_API_KEY')
groq_api_key       = os.getenv('GROQ_API_KEY')
openrouter_api_key = os.getenv('OPENROUTER_API_KEY')

for name, key in [
    ("OpenAI",     openai_api_key),
    ("Anthropic",  anthropic_api_key),
    ("Google",     google_api_key),
    ("Groq",       groq_api_key),
    ("OpenRouter", openrouter_api_key),
]:
    status = f"begins '{key[:8]}...'" if key else "NOT SET"
    print(f"{name}: {status}")

OpenAI: begins 'sk-proj-...'
Anthropic: begins 'sk-ant-a...'
Google: begins 'AIzaSyCE...'
Groq: begins 'gsk_eskK...'
OpenRouter: begins 'sk-or-v1...'


## 2. API Clients & Models

In [12]:
openai_client    = OpenAI()
anthropic_client = OpenAI(api_key=anthropic_api_key,  base_url="https://api.anthropic.com/v1/")
gemini_client    = OpenAI(api_key=google_api_key,     base_url="https://generativelanguage.googleapis.com/v1beta/openai/")
groq_client      = OpenAI(api_key=groq_api_key, base_url="https://api.groq.com/openai/v1")
ollama_client    = OpenAI(api_key="ollama", base_url="http://localhost:11434/v1")
openrouter_client = OpenAI(api_key=openrouter_api_key, base_url="https://openrouter.ai/api/v1")

# Model definitions — swap for cheaper variants if needed
MODELS = {
    "GPT-5.4":           (openai_client,     "gpt-5.4"),
    "Claude Sonnet 4.6":(anthropic_client,  "claude-sonnet-4-6"),
    "Gemini 3.1 Pro": (gemini_client,     "gemini-3.1-pro-preview"),
    "Grok 4":    (openrouter_client, "x-ai/grok-4"),
    "Qwen 3 Coder":    (openrouter_client, "qwen/qwen3-coder-30b-a3b-instruct"),
    "GPT OSS 20B":    (ollama_client, "gpt-oss:20b"),
    "Deepseek Coder V2":    (ollama_client, "deepseek-coder-v2"),
    "Qwen 2.5 Coder":    (ollama_client, "qwen2.5-coder"),
    "GPT OSS 120B":    (groq_client, "openai/gpt-oss-120b"),
}

print("Available models:", list(MODELS.keys()))

Available models: ['GPT-5.4', 'Claude Sonnet 4.6', 'Gemini 3.1 Pro', 'Grok 4', 'Qwen 3 Coder', 'GPT OSS 20B', 'Deepseek Coder V2', 'Qwen 2.5 Coder', 'GPT OSS 120B']


## 3. System Info & Compile Command

Detect the local compiler and determine the optimal compile flags for this machine.

In [3]:
from system_info import retrieve_system_info

system_info = retrieve_system_info()

os_name   = system_info["os"]["system"]
arch      = system_info["os"]["arch"]
compilers = system_info["toolchain"]["compilers"]

print(f"OS: {os_name} | Arch: {arch}")
print(f"Available compilers: { {k:v[:30] for k,v in compilers.items() if v} }")

OS: Darwin | Arch: arm64
Available compilers: {'gcc': 'Apple clang version 17.0.0 (cl', 'g++': 'Apple clang version 17.0.0 (cl', 'clang': 'Apple clang version 17.0.0 (cl'}


In [4]:
# Build compile command based on detected platform
def build_compile_command() -> list[str]:
    """Return the best compile command for the current machine."""
    base = ["g++", "-O3", "-std=c++17", "main.cpp", "-o", "main"]
    if arch == "arm64" and os_name == "Darwin":
        base.insert(4, "-mcpu=apple-m1")
    return base

compile_command = build_compile_command()
run_command     = ["./main"]

print("Compile:", " ".join(compile_command))
print("Run:    ", " ".join(run_command))

Compile: g++ -O3 -std=c++17 main.cpp -mcpu=apple-m1 -o main
Run:     ./main


## 4. Core Code-Generation Functions

In [5]:
SYSTEM_PROMPT = """\
Your task is to convert Python code into high-performance C++ code.
Respond only with C++ code; no explanations except brief inline comments.
The C++ code must produce identical output to the Python code in the shortest possible time.
"""

def user_prompt_for(python_code: str) -> str:
    return f"""\
Port this Python code to C++ for maximum runtime performance.
System: {system_info['os']['system']} {system_info['os']['arch']}
Compiler: {' '.join(compile_command)}
Output will be written to main.cpp and compiled with the command above.
Respond only with C++ code.

```python
{python_code}
```
"""


def port_to_cpp(model_name: str, python_code: str) -> str:
    """Call the selected model and return generated C++ code."""
    client, model_id = MODELS[model_name]
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": user_prompt_for(python_code)},
    ]
    response = client.chat.completions.create(model=model_id, messages=messages)
    cpp = response.choices[0].message.content
    # Strip markdown fences if present
    cpp = cpp.replace("```cpp", "").replace("```c++", "").replace("```", "").strip()
    return cpp


def run_python(code: str) -> str:
    """Execute Python code and return its stdout output."""
    buf = io.StringIO()
    old_stdout, sys.stdout = sys.stdout, buf
    try:
        exec(code, {"__builtins__": __builtins__})
    except Exception as e:
        return f"Error: {e}"
    finally:
        sys.stdout = old_stdout
    return buf.getvalue()


def compile_and_run(cpp_code: str) -> str:
    """Write cpp_code to main.cpp, compile, run, and return output."""
    with open("main.cpp", "w", encoding="utf-8") as f:
        f.write(cpp_code)
    try:
        subprocess.run(compile_command, check=True, text=True, capture_output=True)
        result = subprocess.run(run_command, check=True, text=True, capture_output=True)
        return result.stdout
    except subprocess.CalledProcessError as e:
        return f"Compilation/run error:\n{e.stderr}"

## 5. Benchmark: Pi Calculation

A CPU-bound Python program that computes π via the Leibniz series.  
We'll measure Python runtime, port to C++ with each model, then compare speedups.

In [6]:
pi_code = """\
import time

def calculate(iterations, param1, param2):
    result = 1.0
    for i in range(1, iterations + 1):
        j = i * param1 - param2
        result -= (1 / j)
        j = i * param1 + param2
        result += (1 / j)
    return result

start = time.time()
result = calculate(100_000_000, 4, 1) * 4
elapsed = time.time() - start

print(f"Result: {result:.12f}")
print(f"Execution Time: {elapsed:.6f} seconds")
"""

print("Running Python baseline...")
t0 = time.time()
python_output = run_python(pi_code)
python_time = time.time() - t0

print(python_output)
print(f"[Wall-clock: {python_time:.2f}s]")

Running Python baseline...
Result: 3.141592658589
Execution Time: 11.365088 seconds

[Wall-clock: 11.37s]


## 6. Port to C++ with Each Model & Measure Speedup

In [13]:
results = {}  # model_name -> {cpp, output, elapsed}

for model_name in MODELS:
    print(f"\n{'='*50}")
    print(f"Porting with: {model_name}")
    try:
        cpp_code = port_to_cpp(model_name, pi_code)

        t0 = time.time()
        output = compile_and_run(cpp_code)
        elapsed = time.time() - t0

        results[model_name] = {"cpp": cpp_code, "output": output, "elapsed": elapsed}
        print(output)
        print(f"[C++ wall-clock: {elapsed:.4f}s]")
    except Exception as e:
        results[model_name] = {"error": str(e)}
        print(f"  ERROR: {e}")


Porting with: GPT-5.4
Compilation/run error:
main.cpp:1:10: fatal error: 'bits/stdc++.h' file not found
    1 | #include <bits/stdc++.h>
      |          ^~~~~~~~~~~~~~~
1 error generated.

[C++ wall-clock: 0.5795s]

Porting with: Claude Sonnet 4.6
Result: 3.141592658589
Execution Time: 0.257736 seconds

[C++ wall-clock: 1.7077s]

Porting with: Gemini 3.1 Pro


KeyboardInterrupt: 

## 7. Performance Summary

In [8]:
print(f"Python baseline wall-clock: {python_time:.4f}s\n")
print(f"{'Model':<25} {'C++ Time':>10} {'Speedup':>10} {'Status'}")
print("-" * 60)

ranked = []
for model_name, data in results.items():
    if "error" in data:
        print(f"{model_name:<25} {'N/A':>10} {'N/A':>10}  ERROR")
    elif "Compilation" in data["output"] or "Error" in data["output"]:
        print(f"{model_name:<25} {'FAIL':>10} {'N/A':>10}  Compilation failed")
    else:
        elapsed = data["elapsed"]
        speedup = python_time / elapsed if elapsed > 0 else float("inf")
        ranked.append((model_name, elapsed, speedup))
        print(f"{model_name:<25} {elapsed:>10.4f}s {speedup:>9.0f}x")

if ranked:
    ranked.sort(key=lambda x: x[1])
    winner = ranked[0]
    print(f"\nFastest: {winner[0]} — {winner[2]:.0f}x speedup over Python")

Python baseline wall-clock: 11.3654s

Model                       C++ Time    Speedup Status
------------------------------------------------------------
GPT-4o                        1.9908s         6x
Claude Sonnet 4.6             1.3109s         9x
Gemini 2.5 Flash                 N/A        N/A  ERROR
Grok 3 (fast)                    N/A        N/A  ERROR

Fastest: Claude Sonnet 4.6 — 9x speedup over Python


## 8. Generated C++ Code (Inspect)

Display the best-performing C++ output.

In [9]:
if ranked:
    best_model = ranked[0][0]
    best_cpp   = results[best_model]["cpp"]
    print(f"Best model: {best_model}\n")
    display(Markdown(f"```cpp\n{best_cpp}\n```"))

Best model: Claude Sonnet 4.6



```cpp
#include <iostream>
#include <iomanip>
#include <chrono>

int main() {
    auto start = std::chrono::high_resolution_clock::now();

    const long long iterations = 100000000LL;
    const double param1 = 4.0;
    const double param2 = 1.0;

    double result = 1.0;

    for (long long i = 1; i <= iterations; i++) {
        double j1 = i * param1 - param2;
        double j2 = i * param1 + param2;
        result -= 1.0 / j1;
        result += 1.0 / j2;
    }

    result *= 4.0;

    auto end = std::chrono::high_resolution_clock::now();
    double elapsed = std::chrono::duration<double>(end - start).count();

    std::cout << std::fixed << std::setprecision(12);
    std::cout << "Result: " << result << "\n";
    std::cout << std::setprecision(6);
    std::cout << "Execution Time: " << elapsed << " seconds\n";

    return 0;
}
```

## 9. Interactive Gradio UI

A full-featured UI: paste any Python code, choose a model, convert to C++, then run both and compare outputs side by side.

In [14]:
def gradio_port(model_name: str, python_code: str) -> str:
    """Gradio wrapper: generate C++ from Python."""
    if not python_code.strip():
        return "Please enter Python code."
    try:
        return port_to_cpp(model_name, python_code)
    except Exception as e:
        return f"Error generating C++: {e}"


def gradio_run_python(code: str) -> str:
    if not code.strip():
        return "No code to run."
    return run_python(code)


def gradio_run_cpp(cpp_code: str) -> str:
    if not cpp_code.strip():
        return "No C++ code to compile."
    return compile_and_run(cpp_code)


model_names = list(MODELS.keys())

with gr.Blocks(title="Python → C++ Code Generator") as ui:
    gr.Markdown("## Python → C++ Code Generator")
    gr.Markdown("Enter Python code, choose a model, generate optimized C++, then run both to compare.")

    with gr.Row():
        python_input = gr.Code(label="Python (input)",    language="python", lines=20, value=pi_code)
        cpp_output   = gr.Code(label="C++ (generated)",   language="cpp",    lines=20)

    with gr.Row():
        model_dd   = gr.Dropdown(model_names, value=model_names[0], label="Model")
        run_py_btn = gr.Button("▶ Run Python",  variant="secondary")
        convert_btn = gr.Button("⚡ Convert to C++", variant="primary")
        run_cpp_btn = gr.Button("▶ Compile & Run C++", variant="secondary")

    with gr.Row():
        py_result  = gr.Textbox(label="Python output",  lines=5)
        cpp_result = gr.Textbox(label="C++ output",     lines=5)

    run_py_btn.click( fn=gradio_run_python, inputs=[python_input], outputs=[py_result])
    convert_btn.click(fn=gradio_port,       inputs=[model_dd, python_input], outputs=[cpp_output])
    run_cpp_btn.click(fn=gradio_run_cpp,    inputs=[cpp_output],  outputs=[cpp_result])

ui.launch(inbrowser=True)

* Running on local URL:  http://127.0.0.1:7864
* To create a public link, set `share=True` in `launch()`.
